# 🔬 Thin Film Hardness Prediction using Artificial Neural Networks

**Author:** Veena Sahu (@Veena Sahu)

---

## Project Overview

This project uses an **Artificial Neural Network (ANN)** to predict the **Vickers Hardness (VHN)** of thin film multi-principal element alloys (MPEAs). The model is trained on a curated dataset of 218 alloys using physically meaningful features derived from alloy composition and thermodynamic properties.

### Key Highlights
- **K-Fold Cross-Validation** (K=5) for robust model evaluation
- **Convergence-checked training** — automatic restart if the model gets stuck in a local minimum
- **Comprehensive visualizations** — loss curves, error distributions, actual vs predicted plots

### Features Used
| Feature | Description |
|---------|-------------|
| `R_cov_delta` | Covalent radius mismatch |
| `G_delta` | Shear modulus mismatch |
| `VEC` | Valence electron concentration |
| `E_delta` | Young's modulus mismatch |
| `H_chem` | Chemical enthalpy |
| `H_el` | Elastic strain energy |

---
## 1. Imports & Configuration

In [ ]:
# ── Standard Libraries ──
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # suppress TF logs

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Machine Learning ──
import tensorflow as tf
from tensorflow.keras import layers, Model, optimizers

from sklearn.utils import shuffle
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# ── Plot styling ──
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print(f'TensorFlow version: {tf.__version__}')
print('All imports successful ✓')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Configuration  (replaces the external Input_ANN.txt file)
# ══════════════════════════════════════════════════════════════════

CONFIG = {
    'project_name'          : 'ThinFilm_Hardness_ANN',
    'database'              : 'db_HEAs.csv',
    'target'                : 'VHN',
    'features'              : ['R_cov_delta', 'G_delta', 'VEC', 'E_delta', 'H_chem', 'H_el'],
    'layer_units'           : [20, 15, 10, 1],
    'activation_functions'  : ['sigmoid', 'sigmoid', 'relu', 'relu'],
    'loss_function'         : 'mae',
    'optimizer'             : 'Adam',
    'learning_rate'         : 0.02,
    'iterations'            : 400,
    'save_after_iterations' : 100,
    'check_error'           : 101,
    'check_after_iterations': 200,
    'k_folds'               : 5,
}

print('Configuration loaded ✓')
for k, v in CONFIG.items():
    print(f'  {k:28s}: {v}')

---
## 2. Data Loading & Exploration

In [ ]:
# ── Load the dataset ──
db = pd.read_csv(CONFIG['database'], encoding='latin-1')

# Keep only required columns
cols_needed = ['alloy_name', 'phases', CONFIG['target']] + CONFIG['features']
db = db[cols_needed]

# Drop rows with NaN in target or feature columns
db = db.dropna(axis='index', subset=[CONFIG['target']] + CONFIG['features'])

print(f'Dataset shape: {db.shape}')
print(f'Number of alloys: {len(db)}')
print(f'Target column: {CONFIG["target"]}')
db.head(10)

In [ ]:
# ── Descriptive Statistics ──
db[[CONFIG['target']] + CONFIG['features']].describe().round(4)

In [ ]:
# ── Target Distribution ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(db[CONFIG['target']], bins=30, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Vickers Hardness (VHN)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Hardness')

axes[1].boxplot(db[CONFIG['target']], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#4C72B0', alpha=0.7))
axes[1].set_ylabel('Vickers Hardness (VHN)')
axes[1].set_title('Hardness Box Plot')

plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Correlation Heatmap ──
corr_cols = CONFIG['features'] + [CONFIG['target']]
corr_matrix = db[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# ── Phase Distribution ──
phase_counts = db['phases'].value_counts()
fig, ax = plt.subplots(figsize=(10, 4))
phase_counts.plot(kind='barh', color='#4C72B0', edgecolor='white', ax=ax)
ax.set_xlabel('Count')
ax.set_ylabel('Phase')
ax.set_title('Distribution of Phases in Dataset')
plt.tight_layout()
plt.show()

---
## 3. Data Preparation

In [ ]:
# ── Extract features (X) and target (Y) ──
X = db[CONFIG['features']]
Y = db[[CONFIG['target']]]

# Shuffle dataset
X, Y = shuffle(X, Y, random_state=1)

# Build labels for result tables
phases = np.array(db['phases']).astype('str')
labels = db['alloy_name'] + ' (' + phases + ')'

# Convert to numpy arrays
X = np.array(X)
Y = np.array(Y)

print(f'X shape: {X.shape}')
print(f'Y shape: {Y.shape}')

---
## 4. ANN Model Definition

In [ ]:
def build_ann_model(config):
    """
    Build and compile a Keras Sequential-style ANN for hardness prediction.

    Parameters
    ----------
    config : dict
        Configuration dictionary containing model hyperparameters.

    Returns
    -------
    tf.keras.Model
        Compiled Keras model ready for training.
    """
    n_feats          = len(config['features'])
    layer_units      = config['layer_units']
    layer_activation = config['activation_functions']
    opt_name         = config['optimizer']
    lr               = config['learning_rate']
    loss_fn          = config['loss_function']

    # Input layer
    feature_input = layers.Input(shape=n_feats)
    x = feature_input

    # Hidden layers
    for h in range(len(layer_units) - 1):
        x = layers.Dense(layer_units[h], activation=layer_activation[h])(x)

    # Output layer
    output_layer = layers.Dense(layer_units[-1], activation=layer_activation[-1])(x)

    # Assemble model
    model = Model(feature_input, output_layer)

    # Compile
    opt = getattr(optimizers, opt_name)(learning_rate=lr)
    model.compile(loss=loss_fn, optimizer=opt, metrics=['mae'])

    return model


# ── Preview model architecture ──
demo_model = build_ann_model(CONFIG)
demo_model.summary()
del demo_model

---
## 5. K-Fold Cross-Validation Training

In [ ]:
# ── Create results directory ──
results_dir = os.path.join(CONFIG['project_name'])
model_save_dir = os.path.join(results_dir, 'model_save')
pred_save_dir  = os.path.join(results_dir, 'prediction_save')
perf_save_dir  = os.path.join(results_dir, 'performance_save')

for d in [results_dir, model_save_dir, pred_save_dir, perf_save_dir]:
    os.makedirs(d, exist_ok=True)

print(f'Results will be saved to: {results_dir}/')

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  K-Fold Cross-Validation Training Loop
# ══════════════════════════════════════════════════════════════════

K = CONFIG['k_folds']
L = len(Y)
K_L = int(L / K)  # size of each validation fold
x_feats = CONFIG['features']

iterations            = CONFIG['iterations']
check_after           = CONFIG['check_after_iterations']
save_after            = CONFIG['save_after_iterations']
threshold             = CONFIG['check_error']

# DataFrames to store loss and MAE across all folds
loss_db = pd.DataFrame({'Iteration': np.arange(1, iterations + 1)})
mae_db  = pd.DataFrame({'Iteration': np.arange(1, iterations + 1)})

# Store all fold results for summary
all_fold_results = []

print(f'Running {K}-fold cross-validation ({iterations} iterations per fold)\n')
print('=' * 60)

for Kn in range(K):
    print(f'\n▶ Fold {Kn+1}/{K}')
    print('-' * 40)

    # ── Split data ──
    start_idx = Kn * K_L
    end_idx   = start_idx + K_L

    Kx_train = np.delete(X, np.s_[start_idx:end_idx], axis=0)
    Kx_test  = X[start_idx:end_idx]
    Ky_train = np.delete(Y, np.s_[start_idx:end_idx], axis=0)
    Ky_test  = Y[start_idx:end_idx]
    K_labels = labels[start_idx:end_idx]

    # Convert to DataFrames for column names
    Kx_train_df = pd.DataFrame(Kx_train, columns=x_feats)
    Kx_test_df  = pd.DataFrame(Kx_test,  columns=x_feats)

    # ── Lists for history tracking ──
    train_K_loss, val_K_loss = [], []
    train_K_mae,  val_K_mae  = [], []

    # ── Build model with convergence check ──
    check_err = threshold + 2
    restart_count = 0

    while check_err > threshold:
        ANN_model = build_ann_model(CONFIG)
        ANN_eval = ANN_model.fit(
            Kx_train_df, Ky_train,
            epochs=check_after,
            validation_data=(Kx_test_df, Ky_test),
            verbose=0
        )
        check_err = ANN_eval.history['mae'][-1]
        restart_count += 1

    if restart_count > 1:
        print(f'  ⟳ Model restarted {restart_count - 1} time(s) to escape local minima')

    # Record initial training history
    train_K_loss += ANN_eval.history['loss']
    val_K_loss   += ANN_eval.history['val_loss']
    train_K_mae  += ANN_eval.history['mae']
    val_K_mae    += ANN_eval.history['val_mae']

    # ── Continue training with periodic saves ──
    n_stops = int((iterations - check_after) / save_after)

    perf = pd.DataFrame(columns=['K-fold', 'Iteration', 'MAE_train', 'MAE_test', 'RMSE_test', 'R2_test'])

    for stop in range(1, n_stops + 1):
        ANN_eval = ANN_model.fit(
            Kx_train_df, Ky_train,
            epochs=save_after,
            validation_data=(Kx_test_df, Ky_test),
            verbose=0
        )

        train_K_loss += ANN_eval.history['loss']
        val_K_loss   += ANN_eval.history['val_loss']
        train_K_mae  += ANN_eval.history['mae']
        val_K_mae    += ANN_eval.history['val_mae']

        it_n = check_after + stop * save_after
        print(f'  Iteration {it_n:>5d} — saving checkpoint')

        # Save model
        model_path = os.path.join(model_save_dir, f'fold{Kn+1}_iter{it_n}.h5')
        ANN_model.save(model_path)

        # ── Predictions ──
        Kx_train_r = Kx_train.reshape(Ky_train.shape[0], 1, len(x_feats))
        Kx_test_r  = Kx_test.reshape(Ky_test.shape[0], 1, len(x_feats))

        pred_train = np.array([ANN_model.predict(Kx_train_r[i], verbose=0) for i in range(Ky_train.shape[0])]).reshape(-1, 1)
        pred_test  = np.array([ANN_model.predict(Kx_test_r[i],  verbose=0) for i in range(Ky_test.shape[0])]).reshape(-1, 1)

        # Save predictions
        pred_df = pd.DataFrame({
            'Alloy': K_labels.values,
            'VHN_actual': Ky_test.flatten(),
            'VHN_predicted': pred_test.flatten(),
            'Error': (pred_test - Ky_test).flatten(),
            'Abs_Error': np.abs(pred_test - Ky_test).flatten(),
            'Pct_Error': (np.abs(pred_test - Ky_test) * 100 / Ky_test).flatten()
        })
        pred_df.to_csv(os.path.join(pred_save_dir, f'fold{Kn+1}_iter{it_n}_pred.csv'), index=False)

        # Calculate metrics
        MAE_train = mean_absolute_error(Ky_train, pred_train)
        MAE_test  = mean_absolute_error(Ky_test,  pred_test)
        RMSE_test = mean_squared_error(Ky_test, pred_test) ** 0.5
        r2_test   = r2_score(Ky_test, pred_test)

        perf.loc[stop - 1] = [f'K{Kn+1}', it_n, MAE_train, MAE_test, RMSE_test, r2_test]

    # Save fold performance
    perf.to_csv(os.path.join(perf_save_dir, f'fold{Kn+1}_perf.csv'), index=False)
    all_fold_results.append(perf)

    # Store loss / MAE history
    loss_db[f'K{Kn+1}_train'] = train_K_loss
    loss_db[f'K{Kn+1}_val']   = val_K_loss
    mae_db[f'K{Kn+1}_train']  = train_K_mae
    mae_db[f'K{Kn+1}_val']    = val_K_mae

    print(f'  ✓ Fold {Kn+1} complete — Final MAE (test): {MAE_test:.2f}, R²: {r2_test:.4f}')

# Save aggregated results
loss_db.to_csv(os.path.join(results_dir, 'loss.csv'), index=False)
mae_db.to_csv(os.path.join(results_dir, 'mae.csv'), index=False)

print('\n' + '=' * 60)
print('Training complete ✓')

---
## 6. Results & Visualization

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  6a. Training & Validation Loss Curves (per fold)
# ══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, K, figsize=(4 * K, 4), sharey=True)
if K == 1:
    axes = [axes]

for i in range(K):
    ax = axes[i]
    ax.plot(loss_db['Iteration'], loss_db[f'K{i+1}_train'], label='Train', alpha=0.8)
    ax.plot(loss_db['Iteration'], loss_db[f'K{i+1}_val'],   label='Validation', alpha=0.8)
    ax.set_title(f'Fold {i+1}')
    ax.set_xlabel('Iteration')
    if i == 0:
        ax.set_ylabel('Loss (MAE)')
    ax.legend(fontsize=8)

fig.suptitle('Training & Validation Loss per Fold', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  6b. MAE Curves
# ══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, K, figsize=(4 * K, 4), sharey=True)
if K == 1:
    axes = [axes]

for i in range(K):
    ax = axes[i]
    ax.plot(mae_db['Iteration'], mae_db[f'K{i+1}_train'], label='Train MAE', alpha=0.8)
    ax.plot(mae_db['Iteration'], mae_db[f'K{i+1}_val'],   label='Val MAE', alpha=0.8)
    ax.set_title(f'Fold {i+1}')
    ax.set_xlabel('Iteration')
    if i == 0:
        ax.set_ylabel('MAE')
    ax.legend(fontsize=8)

fig.suptitle('Mean Absolute Error per Fold', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  6c. Actual vs Predicted Scatter Plot (last iteration, all folds)
# ══════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(7, 6))

colors = plt.cm.Set2(np.linspace(0, 1, K))
all_actual, all_pred = [], []

last_iter = check_after + n_stops * save_after

for i in range(K):
    pred_file = os.path.join(pred_save_dir, f'fold{i+1}_iter{last_iter}_pred.csv')
    if os.path.exists(pred_file):
        df = pd.read_csv(pred_file)
        ax.scatter(df['VHN_actual'], df['VHN_predicted'],
                   alpha=0.7, s=40, color=colors[i], edgecolors='white',
                   linewidth=0.5, label=f'Fold {i+1}', zorder=3)
        all_actual.extend(df['VHN_actual'].values)
        all_pred.extend(df['VHN_predicted'].values)

# Perfect prediction line
lims = [min(all_actual + all_pred) - 50, max(all_actual + all_pred) + 50]
ax.plot(lims, lims, 'k--', alpha=0.5, linewidth=1, label='Perfect prediction')
ax.set_xlim(lims)
ax.set_ylim(lims)

# Overall metrics
overall_r2  = r2_score(all_actual, all_pred)
overall_mae = mean_absolute_error(all_actual, all_pred)

ax.set_xlabel('Actual Hardness (VHN)', fontsize=12)
ax.set_ylabel('Predicted Hardness (VHN)', fontsize=12)
ax.set_title(f'Actual vs Predicted Hardness\nR² = {overall_r2:.4f}  |  MAE = {overall_mae:.2f}', fontsize=13)
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  6d. Error Distribution
# ══════════════════════════════════════════════════════════════════

errors = np.array(all_pred) - np.array(all_actual)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(errors, bins=25, color='#DD8452', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_xlabel('Prediction Error (VHN)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Error Distribution')

axes[1].hist(np.abs(errors), bins=25, color='#55A868', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Absolute Error (VHN)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Absolute Error Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  6e. Per-Fold Performance Summary
# ══════════════════════════════════════════════════════════════════

summary = pd.concat(all_fold_results, ignore_index=True)

# Show only the last iteration per fold
final_perf = summary.groupby('K-fold').last().reset_index()
final_perf = final_perf[['K-fold', 'Iteration', 'MAE_train', 'MAE_test', 'RMSE_test', 'R2_test']]

print('\n📊 Final Performance per Fold')
print('=' * 70)
display(final_perf.style.format({
    'MAE_train': '{:.2f}',
    'MAE_test':  '{:.2f}',
    'RMSE_test': '{:.2f}',
    'R2_test':   '{:.4f}'
}).set_properties(**{'text-align': 'center'}))

print(f'\nOverall Mean MAE (test):  {final_perf["MAE_test"].mean():.2f}')
print(f'Overall Mean RMSE (test): {final_perf["RMSE_test"].mean():.2f}')
print(f'Overall Mean R² (test):   {final_perf["R2_test"].mean():.4f}')

---
## 7. Conclusion

This notebook demonstrates a complete pipeline for **predicting thin film hardness** using an Artificial Neural Network:

1. **Data exploration** — visualized target distribution, feature correlations, and phase composition  
2. **ANN model** — multi-layer perceptron with sigmoid/relu activations, trained with Adam optimizer  
3. **5-Fold cross-validation** — ensures robust, unbiased evaluation across the entire dataset  
4. **Convergence-safe training** — automatic model restart if convergence criteria are not met  
5. **Comprehensive results** — loss curves, scatter plots, error distributions, and per-fold metrics  

All trained models, predictions, and performance logs are saved in the `ThinFilm_Hardness_ANN/` directory for further analysis.

---

**Author:** Veena Sahu (@Veena Sahu)  
**Dataset:** 218 multi-principal element alloys with Vickers hardness and 6 physically derived features